# Porting Applications to GPU

The previous hello world example and its GPU port already thought us a lot about GPU programming.
Our next step is formalizing and extending the process we've successfully applied to create some guidelines for future porting efforts.

We start, as usual, by loading our ice magic.

In [ ]:
%load_ext ice.magic

## GPU Memory Management

Apart from specifying code to be executed on the GPU (our kernels) and execution parallelism (our execution configurations), we need to be able to handle data.
For most system architectures used in practice, CPUs and GPUs have **distinct** physical memories which has certain implications:
* Memory can and has to be allocated for CPU and GPU separately.
* Conceptually, the CPU has no access to GPU data and vice versa.
* Data has to be explicitly copied between CPU and GPU allocations.

We will work more with this approach later, but for now we discuss one helpful alternative - **managed memory**.
In this model, allocations are places in unified virtual memory (UVA) and **migrated** on access between GPU and CPU.

## Porting Checklist

0. Verify that the CPU-only applications works as intended.
1. Replace CPU-only allocations with managed memory allocations. Check that everything still works.
2. Identify 'hot loops' suited for GPU offloading and do the following steps for each separately:
    * Extract the loop to a separate function.
    * Convert the function to a kernel using a grid-stride loop.
    * Convert the function call to a kernel launch + synchronize. Start with one thread and check that everything still works.
    * Set a reasonable number of threads and check that everything still works.
3. Profile and optimize. Check that everything still works after each optimization.

## 0. CPU-Only Application

Time for a new application - this time with actual data!
Our new application is still pretty straightforward:
* The basis is an array with `numElements` elements. By default its length is 4M.
* All array elements are initialized with their index plus a fixed constant (1337 in our case).
* For a number of iterations `numIterations` (mimicking an iterative algorithm or time-stepping scheme) ...
    * ... each element of the vector is increase by 1.
* The array elements are checked for correct values (index plus constant plus number of iterations).

Let's have a look at the full code below.

In [ ]:
%%cuda -c code/porting/increase-base.cu

int main(int argc, char *argv[]) {
    size_t numElements = 4 * 1024 * 1024;
    size_t numIterations = 8;

    size_t *data;
    data = new size_t[numElements];

    //# initialize data
    for (int i = 0; i < numElements; ++i) {
        data[i] = i + 1337;
    }

    //# main 'work'
    for (int it = 0; it < numIterations; ++it) {
        for (int i = 0; i < numElements; ++i) {
            data[i] += 1;
        }
    }

    //# verify results
    for (int i = 0; i < numElements; ++i) {
        auto expected = (i + 1337) + numIterations;
        if (data[i] != expected) {
            std::cout << "Error at index " << i << ": got " << data[i] << ", expected " << expected << std::endl;
            break;
        }

        if (numElements - 1 == i)
            std::cout << "Success" << std::endl;
    }

    delete[] data;
}

Since the length of the example has grown quite a bit compared to our hello world application, now is a good time to extract some of the functionality unimportant to our porting effort:
* `initializeData` performs the array element initialization and
* `verifyData` checks that every element holds the correct value after the main computation.

Their implementations are in the [util.h](./code/util.h) header.

In [ ]:
%%cuda -c code/porting/increase-shortened.cu

int main(int argc, char *argv[]) {
    size_t numElements = 4 * 1024 * 1024;
    size_t numIterations = 8;

    size_t *data;
    data = new size_t[numElements];

    initializeData(data, numElements);
        👆

    //# main 'work'
    for (int it = 0; it < numIterations; ++it) {
        for (int i = 0; i < numElements; ++i) {
            data[i] += 1;
        }
    }

    verifyData(data, numElements, numIterations);
        👆

    delete[] data;
}

## 1. Revise Memory Management

Next we use managed memory allocations which are automatically **migrated** on demand between host and device.
These are the the relevant API functions:

```c++
cudaMallocManaged(void** ptr_to_unified_ptr, size_t size_in_bytes)
```
* Allocates a contiguous chunk of managed or unified memory.
* The resulting pointer is stored in `*ptr_to_unified_ptr`, i.e. the function expects a pointer to the pointer to be set as input.
* The allocation size (`size_in_bytes`) is passed in number of bytes and is usually computed with `num_elements * sizeof(T)`.
* [Documentation](https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__MEMORY.html#group__CUDART__MEMORY_1gd228014f19cc0975ebe3e0dd2af6dd1b)

```c++
cudaFree(void* unified_ptr)
```
* Releases the unified allocation from a previous `cudaMallocManaged` call.
* In contrast to the allocation function, this function takes the target pointer directly (instead of a pointer to a pointer).
* [Documentation](https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__MEMORY.html#group__CUDART__MEMORY_1ga042655cbbf3408f01061652a075e094)

## Exercise - Refactor with Managed Memory

Your task is to refactor the below code to use managed memory instead of regular host allocations.
Make sure to also use the matching deallocation.

Keep in mind, that your changes must not change the output of the application.
If it doesn't print `Success` something went wrong.

Bonus task: use error checking to verify that allocating worked as intended.

In [ ]:
%%cuda -c code/porting/ex-malloc-managed.cu

int main(int argc, char *argv[]) {
    size_t numElements = 4 * 1024 * 1024;
    size_t numIterations = 8;

    size_t *data;
    data = new size_t[numElements];
     👆

    initializeData(data, numElements);

    //# main 'work'
    for (int it = 0; it < numIterations; ++it) {
        for (int i = 0; i < numElements; ++i) {
            data[i] += 1;
        }
    }

    verifyData(data, numElements, numIterations);

    delete[] data;
     👆
}

### Possible Solution

In [ ]:
%%cuda -c code/porting/sol-malloc-managed.cu

int main(int argc, char *argv[]) {
    size_t numElements = 4 * 1024 * 1024;
    size_t numIterations = 8;

    size_t *data;
    cudaMallocManaged(&data, numElements * sizeof(size_t));
        👆

    initializeData(data, numElements);

    //# main 'work'
    for (int it = 0; it < numIterations; ++it) {
        for (int i = 0; i < numElements; ++i) {
            data[i] += 1;
        }
    }

    verifyData(data, numElements, numIterations);

    cudaFree(data);
        👆
}

As usual it's a good idea to practice robust code development by consistently using error checks.

In [ ]:
%%cuda -c code/porting/sol-bonus-malloc-managed.cu

int main(int argc, char *argv[]) {
    size_t numElements = 4 * 1024 * 1024;
    size_t numIterations = 8;

    size_t *data;
    checkCudaError(cudaMallocManaged(&data, numElements * sizeof(size_t)));
        👆

    initializeData(data, numElements);

    //# main 'work'
    for (int it = 0; it < numIterations; ++it) {
        for (int i = 0; i < numElements; ++i) {
            data[i] += 1;
        }
    }

    verifyData(data, numElements, numIterations);

    checkCudaError(cudaFree(data));
        👆
}

## 2. Offload 'Hot' Loops

This step would usually entail
* profiling the application to find out where the majority of the execution time is spent, and
* identifying code parts that would benefit from GPU offloading.

For this, we would be looking for
* a high degree of potential parallelism, e.g. loops with millions of iterations that can be **processed independently**,
* **compute-heavy** code parts, and
* operations (loops) that can be done back-to-back to avoid migrating data back and forth.

For our example, however, we focus on the work loop where we repeatedly update array elements.
Note that in practice
* the kind of initialization used should be done on GPU,
* the different iterations should be fused,
* the verification might be done on GPU as well.

We will ignore all of these good practices, and instead focus on porting the loop over the array elements exclusively.

## Exercise - Add Kernels

Look at the slightly adapted code sample below.
Here, the main work loop has already been extracted into the `increase` function.
Your task is to make that operation run on GPU.
Follow these steps:
* Convert the `increase` function to a kernel using the `__global__` keyword.
* Use build-in thread variables to determine
    * a globally unique thread index, and
    * the total number of threads.
* Use a grid-stride loop to partition the work to be done onto the threads.
* Add suitable synchronization with the GPU.
* Determine a suitable execution configuration and launch the new `increase` kernel.
    * Start with a single thread/ block, then
    * switch to a suitable execution configuration, e.g.
        * 256 threads per block and `4 * 1024 * 1024 / 256` blocks, or
        * 2688 (84 * 32) blocks and 256 threads per block - we will discuss where these numbers come from later in [GPU architecture](./04-gpu-architecture.ipynb).
* Verify that the application still prints the `Success` message.

Bonus task: use suitable error checks for synchronous and asynchronous errors.

In [ ]:
%%cuda -c code/porting/ex-kernel.cu

void increase(size_t* data, size_t numElements) {
    for (int i = 0; i < numElements; ++i) {
        data[i] += 1;
    }
}

int main(int argc, char *argv[]) {
    size_t numElements = 4 * 1024 * 1024;
    size_t numIterations = 8;

    size_t *data;
    cudaMallocManaged(&data, numElements * sizeof(size_t));

    initializeData(data, numElements);

    //# main 'work'
    for (int it = 0; it < numIterations; ++it) {
        increase(data, numElements);
    }

    verifyData(data, numElements, numIterations);

    cudaFree(data);
}

### Possible Solution

In [ ]:
%%cuda -c code/porting/sol-kernel.cu

__global__ void increase(size_t* data, size_t numElements) {
    size_t start = blockIdx.x * blockDim.x + threadIdx.x;
    size_t stride = blockDim.x * gridDim.x;
    
    for (size_t i = start; i < numElements; i += stride) {
        data[i] += 1;
    }
}

int main(int argc, char *argv[]) {
    size_t numElements = 4 * 1024 * 1024;
    size_t numIterations = 8;

    size_t *data;
    checkCudaError(cudaMallocManaged(&data, numElements * sizeof(size_t)));

    initializeData(data, numElements);

    //# main 'work'
    for (int it = 0; it < numIterations; ++it) {
        auto numBlocks = 84 * 32;
        auto numThreadsPerBlock = 256;
        increase<<<numBlocks, numThreadsPerBlock>>>(data, numElements);
    }
    checkCudaError(cudaDeviceSynchronize(), true);

    verifyData(data, numElements, numIterations);

    checkCudaError(cudaFree(data));
}

## 3. Profile and Optimize

We start our optimization effort by profiling our application with Nsight Systems.
This can be done on the command line, e.g. with
```bash
nsys profile --stats=true ./my-app my-params
```
or using our ICE magic by providing the `-p` or `--profile` argument.

In [ ]:
%%cuda -c code/porting/profile-kernel.cu -v -p

__global__ void increase(size_t* data, size_t numElements) {
    size_t start = blockIdx.x * blockDim.x + threadIdx.x;
    size_t stride = blockDim.x * gridDim.x;
    
    for (size_t i = start; i < numElements; i += stride) {
        data[i] += 1;
    }
}

int main(int argc, char *argv[]) {
    size_t numElements = 4 * 1024 * 1024;
    size_t numIterations = 8;

    size_t *data;
    checkCudaError(cudaMallocManaged(&data, numElements * sizeof(size_t)));

    initializeData(data, numElements);

    //# main 'work'
    for (int it = 0; it < numIterations; ++it) {
        auto numBlocks = 84 * 32;
        auto numThreadsPerBlock = 256;
        increase<<<numBlocks, numThreadsPerBlock>>>(data, numElements);
    }
    checkCudaError(cudaDeviceSynchronize(), true);

    verifyData(data, numElements, numIterations);

    checkCudaError(cudaFree(data));
}

Wow, that's a lot of information emitted by the profiler.
Let's focus on the `cuda_gpu_kern_sum` section first.
It displays which kernels were executed, how often and what time it took.
Since we only have a single kernel (`increase`), it's the only one shown here and, unsurprisingly, it takes 100% of the execution time of all kernels aggregated.
We can also see that the minimum and maximum runtimes vary strongly between around $10^5$ ns and $10^7$ ns.

### Simple Performance Model

Before diving deeper into the strong variations in execution time per kernel invocation, we want to get a better understanding of whether the time reported is 'good'.
To answer that question, we set up a simplistic performance model.

Generally speaking, performance is either limited by
* the time it takes to do the meaningful computations (compute bound), or
* the time it takes to get the data to and from where the compute is happening (memory bound).

In our test application, we have

```c++
for (size_t i = 0; i < numElements; ++i) {
    data[i] = data[i] + 1;
}
```

Assuming `numElements = 4 * 1024 * 1024`, we have
* 4 M elements and for each element we
* read 8 bytes, do 1 FLOP, write 8 bytes.

This corresponds to a total data volume of 16 B $\cdot$ 4 M = 64 MB and a total compute of 4 MFLOP. \
Ignoring a lot of details, we assume that at **16 bytes per flop** our code is most likely memory bound.

This allows us to compute a **light-speed estimate (LSE)**, an execution time that serves as theoretical limit imposed by hardware characteristics.
The theoretical main memory bandwidth limit of an A40 GPU is 696 GB/s.
This means that our LSE for the execution time is given by 64 MB divided by 696 GB/s.

In [ ]:
lse = 64 / (696 * 1024)
print(f'{lse} s / {lse * 1e3} ms / {lse * 1e9} ns')

So around 90 000 ns - pretty close to our measured **minimum** execution time of 100 000 ns.

Computing the effective bandwidth for the measured **maximum** execution time results in around 6.4 GB/s.
Our next step is figuring out what is going on.

### Visual Profiling

To do so, we visualize the application-level behavior of our program with Nsight Systems GUI following these steps:
1. Profile the applications with `nsys` using the cell below.
2. Locate the profile file in the `profiles` directory.
3. Download the file [profiles/porting-kernel.nsys-rep](profiles/porting-kernel.nsys-rep) (middle click to download or use the file browser).
4. Open the downloaded file locally with Nsight Systems.

In [ ]:
%%cuda -c code/porting/profile-kernel.cu -v -p -o profiles/porting-kernel

__global__ void increase(size_t* data, size_t numElements) {
    size_t start = blockIdx.x * blockDim.x + threadIdx.x;
    size_t stride = blockDim.x * gridDim.x;
    
    for (size_t i = start; i < numElements; i += stride) {
        data[i] += 1;
    }
}

int main(int argc, char *argv[]) {
    size_t numElements = 4 * 1024 * 1024;
    size_t numIterations = 8;

    size_t *data;
    checkCudaError(cudaMallocManaged(&data, numElements * sizeof(size_t)));

    initializeData(data, numElements);

    //# main 'work'
    for (int it = 0; it < numIterations; ++it) {
        auto numBlocks = 84 * 32;
        auto numThreadsPerBlock = 256;
        increase<<<numBlocks, numThreadsPerBlock>>>(data, numElements);
    }
    checkCudaError(cudaDeviceSynchronize(), true);

    verifyData(data, numElements, numIterations);

    checkCudaError(cudaFree(data));
}

## Exercise: Explore Application Timeline

Your task is to follow the steps above and then explore the interactive timeline visualization.
To navigate you can:
* use `ctrl` + `mouse wheel` to zoom in or out,
* `click` and `drag` to mark regions of interest and then
    * `right-click` and select `zoom into selection` or
    * use the shortcut `shift` + `z`,
* `right-click` and select
    * `undo zoom` or
    * `reset zoom`,
* use your `second mouse wheel` (if available) to scroll horizontally.

Start by
* expanding the `CUDA HW` row, and the rows within, by clicking the small triangles next to them,
* locate the part of the timeline where kernels are executed, and
* investigate the kernel and memory operations by hovering over them to show more information as tooltips.

Answer the following questions:
* Are all kernel executions equally fast?
* When is memory transferred
    * from host to device, and
    * from device to host?
* How much time is spent in the `cudaDeviceSynchronize` call? Hint: check the `CUDA API` row.

### Possible Solution

Navigating to the region of interest, we can see the following effects:
* The first kernel invocation takes a lot longer than the subsequent ones.
* While the first kernel is running, memory operations are happening concurrently.
* Hovering over the transfers reveals HtoD (host-to-device) transfers triggered by **page faults**.
* Device-to-host transfers only happen after the bulk computation phase.
* `cudaDeviceSynchronize` takes around 12 ms which is consistent with the information from the earlier CLI analysis.

## Managed Memory and Page Faulting

To understand what we observed in the previous exercise, we need to take a closer look at how managed memory is implemented.

<img align="right" src="img/managed-mem-allocation.png" alt="Managed Memory Allocation" width="640px"/>

Managed memory allocations are broken up into one or more **memory pages**.

<img align="right" src="img/managed-mem-page-fault.png" alt="Managed Memory Page Fault" width="640px"/>

When CPU or GPU parts of a program access memory the corresponding page is deducted from the memory address.
If the requested page is not available, a **page fault** is triggered.

<img align="right" src="img/managed-mem-migration.png" alt="Managed Memory Migration" width="640px"/>

This page fault triggers a **migration** of the affected page to where the access originates from (CPU or GPU).
Note that the CUDA runtime may transfer additional data as part of **speculative prefetching**.

While the process of page faulting and automatic data migration makes our job as developers easier, it comes at the cost of decreased performance due to many small and fragmented data transfers.
One remedy is telling the CUDA runtime where data will be needed.
This can be done through **manual prefetching**.

```c++
cudaMemPrefetchAsync(void* unified_ptr, size_t size_in_bytes, int location)
```
* Prefetches `size_in_bytes` bytes from the `unified_ptr` to the specified `location`.
* Location may be
	* a device index, or
	* `cudaCpuDeviceId` to denote prefetching to host.
* [Documentation](https://docs.nvidia.com/cuda/archive/12.9.1/cuda-c-programming-guide/index.html#data-prefetching)

To get the device Id of the currently active GPU `cudaGetDevice` can be used ([Documentation](https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__DEVICE.html#group__CUDART__DEVICE_1g80861db2ce7c29b6e8055af8ae01bc78))
```c++
int device;
cudaGetDevice(&device);
```

With CUDA 13, the API for prefetching changed to
```c++
cudaMemPrefetchAsync(const void* unified_ptr, size_t size_in_bytes, cudaMemLocation location, unsigned int flags)
```
* Prefetches `size_in_bytes` bytes from the `unified_ptr` to the specified `location` (as before).
* `location` is of type struct `cudaMemLocation` with the two members
	* `int id`
	* `enumcudaMemLocationType type`
* `flags` is currently unused and must be zero.
* [Documentation](https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__MEMORY.html#group__CUDART__MEMORY_1g856fa41c8c0d28655e37b778cb9ffc65)

Examples for the `location` are
```c++
cudaMemLocation location;
location.type = cudaMemLocationTypeDevice;  // type -> device
location.id   = device;                     // id   -> device id as before
```
and
```c++
cudaMemLocation location;
location.type = cudaMemLocationTypeHost;    // type -> host
                                            // id   -> not required/ ignored
```
Note that the `id` is ignored for `cudaMemLocationTypeHost`.
If prefetching to a specific part of the CPU memory is required, `cudaMemLocationTypeHostNuma` can be used in conjunction with an `id`.

## Exercise: Optimize with Manual Prefetching

Have a look at the code below and do the following:

* Add **host-to-device** prefetching at the marked code location.
* Profile the application and check for
    * the kernel execution times on the command line, and
    * the memory transfers on command line and in the timeline view.
* Compare the profiling results ([porting-ex-prefetch.nsys-rep](profiles/porting-ex-prefetch.nsys-rep)) with the previous code version.
* Bonus: add device-to-host prefetching and redo the profiling.

In [ ]:
%%cuda -c code/porting/ex-prefetch.cu -v -p -o profiles/porting-ex-prefetch

__global__ void increase(size_t* data, size_t numElements) {
    size_t start = blockIdx.x * blockDim.x + threadIdx.x;
    size_t stride = blockDim.x * gridDim.x;
    
    for (size_t i = start; i < numElements; i += stride) {
        data[i] += 1;
    }
}

int main(int argc, char *argv[]) {
    size_t numElements = 4 * 1024 * 1024;
    size_t numIterations = 8;

    size_t *data;
    checkCudaError(cudaMallocManaged(&data, numElements * sizeof(size_t)));

    initializeData(data, numElements);

    //# TODO: prefetch data to the GPU
        👆

    //# main 'work'
    for (int it = 0; it < numIterations; ++it) {
        auto numBlocks = 84 * 32;
        auto numThreadsPerBlock = 256;
        increase<<<numBlocks, numThreadsPerBlock>>>(data, numElements);
    }
    checkCudaError(cudaDeviceSynchronize(), true);

    //# TODO: Bonus: prefetch data back to the CPU
        👆

    verifyData(data, numElements, numIterations);

    checkCudaError(cudaFree(data));
}

### Possible Solution

In [ ]:
%%cuda -c code/porting/sol-prefetch.cu -v -p -o profiles/porting-sol-prefetch

__global__ void increase(size_t* data, size_t numElements) {
    size_t start = blockIdx.x * blockDim.x + threadIdx.x;
    size_t stride = blockDim.x * gridDim.x;
    
    for (size_t i = start; i < numElements; i += stride) {
        data[i] += 1;
    }
}

int main(int argc, char *argv[]) {
    size_t numElements = 4 * 1024 * 1024;
    size_t numIterations = 8;

    size_t *data;
    checkCudaError(cudaMallocManaged(&data, numElements * sizeof(size_t)));

    initializeData(data, numElements);

    int device;
    cudaGetDevice(&device);

    //# old version: checkCudaError(cudaMemPrefetchAsync(data, numElements * sizeof(size_t), device));
    checkCudaError(cudaMemPrefetchAsync(data, numElements * sizeof(size_t), { .type = cudaMemLocationTypeDevice, .id = device }, 0));
                    👆

    //# main 'work'
    for (int it = 0; it < numIterations; ++it) {
        auto numBlocks = 84 * 32;
        auto numThreadsPerBlock = 256;
        increase<<<numBlocks, numThreadsPerBlock>>>(data, numElements);
    }
    checkCudaError(cudaDeviceSynchronize(), true);

    //# old version: checkCudaError(cudaMemPrefetchAsync(data, numElements * sizeof(size_t), cudaCpuDeviceId));
    checkCudaError(cudaMemPrefetchAsync(data, numElements * sizeof(size_t), { .type = cudaMemLocationTypeHost }, 0));
                    👆

    verifyData(data, numElements, numIterations);

    checkCudaError(cudaFree(data));
}

## Explicit CUDA Memory APIs

Prefetching allows optimizing the performance of applications relying on managed memory.
In many cases, however, even more control is required over where memory is allocated and how data is moved.
For these cases, explicit allocations can be a remedy as they let you pick where data lives, when it moves, and when allocated resources are released.

In contrast to before, we now have two allocation (and respectively release) functions - one for host memory and one for device memory.
Let's start with the **device** versions.

```c++
cudaMalloc(void** ptr_to_device_ptr, size_t size_in_bytes)
```
* Allocates a contiguous chunk in **device** global memory.
* The resulting pointer is stored in `*ptr_to_device_ptr`, i.e. the function expects a pointer to the pointer to be set as input.
* The allocation size (`size_in_bytes`) is passed in number of bytes and is usually computed with `num_elements * sizeof(T)`.
* [Documentation](https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__MEMORY.html#group__CUDART__MEMORY_1g37d37965bfb4803b6d4e59ff26856356)

```c++
cudaFree(void* device_ptr)
```
* Releases the device allocation from a previous `cudaMalloc` call.
* In contrast to the allocation function, this function takes the target pointer directly (instead of a pointer to a pointer).
* [Documentation](https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__MEMORY.html#group__CUDART__MEMORY_1ga042655cbbf3408f01061652a075e094)

Allocating and releasing **host** memory works similarly using the below functions.

```c++
cudaMallocHost(void** ptr_to_host_ptr, size_t size_in_bytes)
```
* Allocates a contiguous pinned (page-locked) **host** buffer.
* The pointer and size passed follow the same mechanic as with `cudaMalloc`.
* [Documentation](https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__MEMORY.html#group__CUDART__MEMORY_1gab84100ae1fa1b12eaca660207ef585b)


```c++
cudaFreeHost(void* host_ptr)
```
* releases the host allocation from a previous `cudaMallocHost` call.
* [Documentation](https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__MEMORY.html#group__CUDART__MEMORY_1g71c078689c17627566b2a91989184969)

Once multiple allocations have been done, data can be **copied** between them with different CUDA API functions.

```c++
cudaMemcpy(void* destination, const void* source, size_t size_in_bytes, cudaMemcpyKind transfer_direction)
```
* Copies `size_in_bytes` bytes from the `source` to the `destination`.
* The `transfer_direction` ([documentation](https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html#group__CUDART__TYPES_1g18fa99055ee694244a270e4d5101e95b)) may be
  * `cudaMemcpyHostToDevice`,
  * `cudaMemcpyDeviceToHost`,
  * `cudaMemcpyHostToHost`,
  * `cudaMemcpyDeviceToDevice`, or
  * `cudaMemcpyDefault`.
* [Documentation](https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__MEMORY.html#group__CUDART__MEMORY_1gc263dbe6574220cc776b45438fc351e8)

Note: regular host allocations (e.g. via `malloc` or `new`) work as an alternative to `cudaMallocHost`, but yield allocations that are not pinned.
Copying from or to non-pinned memory is less performant.

## Exercise: Use Separate Host and Device Allocations

Refactor the below example to use separate host and device allocations instead of managed memory ones:
* Replace the allocation and release function with their host and device counter-parts.
* Replace the prefetching with copy operations.
* Make sure to pass the correct pointer to the kernel.
* Verify that the performance (bandwidth) is still comparable to the previous code version ([profiles/porting-ex-separate-alloc.nsys-rep](profiles/porting-ex-separate-alloc.nsys-rep)) .

In [ ]:
%%cuda -c code/porting/ex-separate-alloc.cu -v -p -o profiles/porting-ex-separate-alloc

__global__ void increase(size_t* data, size_t numElements) {
    size_t start = blockIdx.x * blockDim.x + threadIdx.x;
    size_t stride = blockDim.x * gridDim.x;
    
    for (size_t i = start; i < numElements; i += stride) {
        data[i] += 1;
    }
}

int main(int argc, char *argv[]) {
    size_t numElements = 4 * 1024 * 1024;
    size_t numIterations = 8;

    size_t *data;
    checkCudaError(cudaMallocManaged(&data, numElements * sizeof(size_t)));

    initializeData(data, numElements);

    int device;
    cudaGetDevice(&device);

    checkCudaError(cudaMemPrefetchAsync(data, numElements * sizeof(size_t), { .type = cudaMemLocationTypeDevice, .id = device }, 0));

    //# main 'work'
    for (int it = 0; it < numIterations; ++it) {
        auto numBlocks = 84 * 32;
        auto numThreadsPerBlock = 256;
        increase<<<numBlocks, numThreadsPerBlock>>>(data, numElements);
    }
    checkCudaError(cudaDeviceSynchronize(), true);

    checkCudaError(cudaMemPrefetchAsync(data, numElements * sizeof(size_t), { .type = cudaMemLocationTypeHost }, 0));

    verifyData(data, numElements, numIterations);

    checkCudaError(cudaFree(data));
}

### Possible Solution

In [ ]:
%%cuda -c code/porting/sol-separate-alloc.cu -v -p -o profiles/porting-sol-separate-alloc

__global__ void increase(size_t* data, size_t numElements) {
    size_t start = blockIdx.x * blockDim.x + threadIdx.x;
    size_t stride = blockDim.x * gridDim.x;
    
    for (size_t i = start; i < numElements; i += stride)
        data[i] += 1;
}

int main(int argc, char *argv[]) {
    size_t numElements = 4 * 1024 * 1024;
    size_t numIterations = 8;

    size_t *data;
    checkCudaError(cudaMallocHost(&data, numElements * sizeof(size_t)));
                      👆
    size_t *d_data;
    checkCudaError(cudaMalloc(&d_data, numElements * sizeof(size_t)));
                      👆

    initializeData(data, numElements);

    //# copy data to device
    checkCudaError(cudaMemcpy(d_data, data, numElements * sizeof(size_t), cudaMemcpyHostToDevice));
                      👆

    //# main 'work'
    for (int it = 0; it < numIterations; ++it) {
        auto numBlocks = 108 * 32;
        auto numThreadsPerBlock = 256;
        increase<<<numBlocks, numThreadsPerBlock>>>(d_data, numElements);
                                                    👆
    }
    checkCudaError(cudaDeviceSynchronize());

    //# copy data back to host
    checkCudaError(cudaMemcpy(data, d_data, numElements * sizeof(size_t), cudaMemcpyDeviceToHost));
                       👆

    verifyData(data, numElements, numIterations);

    checkCudaError(cudaFree(d_data));
                       👆
    checkCudaError(cudaFreeHost(data));
                       👆
}

## Next Step

To continue with the course, head over to the [GPU architecture](./04-gpu-architecture.ipynb) notebook.